<h1 style="
    background-color: #d0ebff;
    color: #1a1a1a;
    display: inline-block;
    padding: 10px 18px;
    border-radius: 10px;
    font-size: 32px;
">
Feature Engineering
</h1>


En esta sección se agregan nuevas variables construidas a partir de las features ya preprocesadas. El objetivo no es sumar columnas por sumar, sino capturar relaciones que el modelo puede estar perdiendo cuando ve cada variable por separado.

A partir del análisis exploratorio y de las matrices de correlación, se priorizan variables relacionadas con antigüedad, uso del vehículo, cilindrada y la combinación entre marca y modelo. En cambio, `Color` no se toma como primer candidato porque su relación individual con `Precio` es débil en esta etapa.


In [ ]:
import pandas as pd
import numpy as np
import sys
import importlib


In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
sys.path.append("../src")

import eda_utils as eda
import preprocessing as prep
import visualization as visual
import modeling as mod

importlib.reload(eda)
importlib.reload(prep)
importlib.reload(visual)
importlib.reload(mod)


<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
Data Loading
</h3>


In [ ]:
X_train = pd.read_csv("../data/X_train.csv")
X_val = pd.read_csv("../data/X_val.csv")
y_train = pd.read_csv("../data/y_train.csv").squeeze()
y_val = pd.read_csv("../data/y_val.csv").squeeze()

X_train.shape, X_val.shape


<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
Feature Criteria
</h3>


#### Features iniciales propuestas

- **`Antigüedad`**: transforma `Año` en una medida más directa de depreciación. En las correlaciones, `Año` aparece asociado positivamente con el precio, por lo que la antigüedad debería capturar el efecto inverso: autos más viejos tienden a valer menos.
- **`Kilómetros_por_año`**: combina `Kilómetros` y `Año`. Esto ayuda a distinguir autos con mucho uso para su edad de autos que, aunque sean antiguos, fueron poco usados.
- **`Es_0km`**: identifica publicaciones con kilometraje nulo o muy bajo. Estos vehículos pueden comportarse distinto a los usados, incluso si comparten marca, modelo y año.
- **`Marca_Modelo`**: captura la interacción entre marca y modelo. Las matrices muestran que muchas marcas están fuertemente asociadas a ciertos modelos, y esa combinación suele ser más informativa que mirar ambas variables por separado.
- **`Es_Alta_Gama`**: resume un segmento de marcas con precios estructuralmente más altos. Esto permite que el modelo tenga una señal general de gama sin depender únicamente de cada dummy individual de marca.
- **`Cilindrada_missing`**: conserva información sobre los casos donde no se pudo extraer cilindrada. Además, permite imputar la cilindrada sin perder el dato de que originalmente faltaba.


<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
Feature Functions
</h3>


In [ ]:
REFERENCE_YEAR = 2025
ZERO_KM_THRESHOLD = 100
PREMIUM_BRANDS = [
    "alfa romeo",
    "audi",
    "bmw",
    "land rover",
    "mercedes benz",
    "porsche",
    "volvo",
]


In [ ]:
def add_usage_features(df, year_col="Año", km_col="Kilómetros", reference_year=REFERENCE_YEAR, zero_km_threshold=ZERO_KM_THRESHOLD):
    """
    Adds vehicle age, kilometers-per-year and zero-kilometer indicators.

    Arguments:
        df (pd.DataFrame): dataset containing year and kilometer columns
        year_col (str): vehicle year column
        km_col (str): kilometers column
        reference_year (int): year used to compute vehicle age
        zero_km_threshold (int | float): maximum kilometers considered as 0km

    Returns:
        pd.DataFrame: dataset with usage-related features
    """
    data = df.copy()

    data[year_col] = pd.to_numeric(data[year_col], errors="coerce")
    data[km_col] = pd.to_numeric(data[km_col], errors="coerce")

    data["Antigüedad"] = (reference_year - data[year_col]).clip(lower=0)
    age_for_ratio = data["Antigüedad"].replace(0, 1)

    data["Kilómetros_por_año"] = data[km_col] / age_for_ratio
    data["Es_0km"] = (data[km_col] <= zero_km_threshold).astype(int)

    return data


In [ ]:
def add_brand_model_feature(train_df, val_df, brand_col="Marca", model_col="Modelo", min_count=20):
    """
    Adds a grouped brand-model interaction feature using only train frequencies.

    Rare combinations are grouped as 'other' to avoid creating many sparse
    columns and to keep validation aligned with training categories.

    Arguments:
        train_df (pd.DataFrame): training dataset
        val_df (pd.DataFrame): validation dataset
        brand_col (str): brand column
        model_col (str): model column
        min_count (int): minimum train frequency required to keep a combination

    Returns:
        tuple[pd.DataFrame, pd.DataFrame]: train and validation datasets with
        the brand-model feature added
    """
    train_data = train_df.copy()
    val_data = val_df.copy()

    train_combo = train_data[brand_col].astype(str) + "_" + train_data[model_col].astype(str)
    val_combo = val_data[brand_col].astype(str) + "_" + val_data[model_col].astype(str)

    frequent_combos = train_combo.value_counts()
    frequent_combos = frequent_combos[frequent_combos >= min_count].index

    train_data["Marca_Modelo"] = train_combo.where(train_combo.isin(frequent_combos), "other")
    val_data["Marca_Modelo"] = val_combo.where(val_combo.isin(frequent_combos), "other")

    return train_data, val_data


In [ ]:
def add_premium_brand_feature(df, premium_brands=PREMIUM_BRANDS, brand_col="Marca"):
    """
    Adds a binary indicator for premium or high-end brands.

    Arguments:
        df (pd.DataFrame): dataset containing the brand column
        premium_brands (list[str]): brand names treated as high-end
        brand_col (str): brand column

    Returns:
        pd.DataFrame: dataset with the premium-brand indicator
    """
    data = df.copy()
    data["Es_Alta_Gama"] = data[brand_col].isin(premium_brands).astype(int)

    return data


In [ ]:
def add_cilindrada_missing_indicator(train_df, val_df, cilindrada_col="Cilindrada"):
    """
    Adds a missingness indicator for engine displacement and imputes missing values.

    The imputation value is computed from the training set only, so validation
    does not leak information into preprocessing.

    Arguments:
        train_df (pd.DataFrame): training dataset
        val_df (pd.DataFrame): validation dataset
        cilindrada_col (str): engine displacement column

    Returns:
        tuple[pd.DataFrame, pd.DataFrame]: train and validation datasets with
        the missingness indicator and imputed displacement
    """
    train_data = train_df.copy()
    val_data = val_df.copy()

    train_data[cilindrada_col] = pd.to_numeric(train_data[cilindrada_col], errors="coerce")
    val_data[cilindrada_col] = pd.to_numeric(val_data[cilindrada_col], errors="coerce")

    train_data[f"{cilindrada_col}_missing"] = train_data[cilindrada_col].isna().astype(int)
    val_data[f"{cilindrada_col}_missing"] = val_data[cilindrada_col].isna().astype(int)

    train_median = train_data[cilindrada_col].median()
    train_data[cilindrada_col] = train_data[cilindrada_col].fillna(train_median)
    val_data[cilindrada_col] = val_data[cilindrada_col].fillna(train_median)

    return train_data, val_data


<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
Applying Features
</h3>


In [ ]:
X_train_fe = add_usage_features(X_train)
X_val_fe = add_usage_features(X_val)

X_train_fe, X_val_fe = add_brand_model_feature(
    X_train_fe,
    X_val_fe,
    min_count=20,
)

X_train_fe = add_premium_brand_feature(X_train_fe)
X_val_fe = add_premium_brand_feature(X_val_fe)

X_train_fe, X_val_fe = add_cilindrada_missing_indicator(X_train_fe, X_val_fe)


In [ ]:
new_features = [
    "Antigüedad",
    "Kilómetros_por_año",
    "Es_0km",
    "Marca_Modelo",
    "Es_Alta_Gama",
    "Cilindrada_missing",
]

X_train_fe[new_features].head()


<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
Encoding
</h3>


Para comparar estos features con los modelos actuales, se aplica el mismo esquema general de encoding. La nueva variable `Marca_Modelo` se trata como categórica y se codifica con one-hot encoding, pero sus categorías se aprenden solo en train.


In [ ]:
columns_to_drop = [
    "Título",
    "Descripción",
    "Versión",
]

categorical_cols_to_encode = [
    "Marca",
    "Modelo",
    "Marca_Modelo",
    "Color",
    "Tipo de vendedor",
    "Tipo de combustible",
    "Transmisión",
]

binary_missing_cols = [
    "Con cámara de retroceso",
]

X_train_fe_model = prep.drop_irrelevant_columns(X_train_fe, columns_to_drop)
X_val_fe_model = prep.drop_irrelevant_columns(X_val_fe, columns_to_drop)

X_train_fe_encoded, categories_map = prep.one_hot_encoding(
    X_train_fe_model,
    categorical_cols=categorical_cols_to_encode,
    train=True,
    binary_missing_cols=binary_missing_cols,
)

X_val_fe_encoded = prep.one_hot_encoding(
    X_val_fe_model,
    categorical_cols=categorical_cols_to_encode,
    train=False,
    categories_map=categories_map,
    binary_missing_cols=binary_missing_cols,
)

X_train_fe_encoded.shape, X_val_fe_encoded.shape


<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
Checking Feature Dataset
</h3>


In [ ]:
feature_types = eda.feature_types_summary(X_train_fe_encoded)
display(feature_types.style.hide(axis="index"))


In [ ]:
missing_summary = eda.missing_values_summary(X_train_fe_encoded)
display(missing_summary.style.hide(axis="index"))


In [ ]:
train_plot_data = visual.build_plot_dataset(X_train_fe_encoded, y_train)

selected_cols = [
    "Precio",
    "Antigüedad",
    "Kilómetros_por_año",
    "Es_0km",
    "Es_Alta_Gama",
    "Cilindrada",
    "Cilindrada_missing",
]

visual.plot_numeric_correlation_heatmap(
    train_plot_data,
    numeric_cols=selected_cols,
    add_log_price=False,
    include_target=False,
    include_log_target=False,
    title="Correlación de nuevas features con Precio",
)


<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
Optional Model Check
</h3>


Esta celda permite comparar rápidamente contra el modelo de referencia. La idea es usarla como chequeo inicial: si las métricas no mejoran, conviene revisar outliers o el tratamiento de marca-modelo antes de seguir agregando features.


In [ ]:
# xgboost_fe_model, xgboost_fe_metrics, xgboost_fe_predictions = mod.train_xgboost(
#     X_train_fe_encoded,
#     y_train,
#     X_val_fe_encoded,
#     y_val,
#     apply_log=True,
# )
#
# visual.plot_regression_metrics(
#     xgboost_fe_metrics,
#     model_name="XGBoost Log - Feature Engineering",
# )
#
# xgboost_fe_metrics


<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
Optional Save
</h3>


Por ahora no se guardan archivos automáticamente para evitar pisar los datasets preprocesados actuales. Si después de evaluar las métricas estos features resultan útiles, se pueden guardar como una nueva versión de train y validation.


In [ ]:
# X_train_fe_encoded.to_csv("../data/X_train_fe.csv", index=False)
# X_val_fe_encoded.to_csv("../data/X_val_fe.csv", index=False)
